In [1]:
import pandas as pd
import geopandas as gpd

import pandas as pd
import numpy as np 
import geopandas as gpd
from census import Census
from us import states
import matplotlib.pyplot as plt

CENSUS_KEY = "7df7c9bb2d367482d3163283c4b4e701841c20a7"
# Initialize the API
c = Census(CENSUS_KEY)

In [2]:
df = pd.read_csv('/Users/madilore/Downloads/Affordable_Housing_Production_by_Building_20260305.csv')


In [3]:
# Let's pull some data from the census again
census_variables = {'B01001_026E': 'Female',
                     'B03002_003E': 'White',
                     'B03002_004E': 'Black',
                     'B03001_003E': 'Hispanic',
                     'B03002_006E': 'Asian',
                     'B06011_001E': 'MedIncome',
                     'B01003_001E': 'TotalPop',
                     'B10059_002E': 'Poverty',
                     'B15003_022E': 'Bach',
                     'B06009_006E': 'GradPro',
                     'B07001_005E': 'Age2024',
                     'B07001_006E': 'Age2529',
                     'B07001_007E': 'Age3034',
                     'B08006_003E': 'DroveAlone',
                     'B08006_008E': 'Transit',
                     'B08006_001E': 'Commuters',
                     'B25106_024E': 'Renter',
                     'B25002_003E': 'Vacant',
                     'B25002_001E': 'TotalUnits',
                     'B25024_002E': 'Singleh',
                     'B25024_001E': 'TotalHouse',
                     'B25097_001E': 'MedHvalue'}

chi_tracts = c.acs5.get( # start here each time
    list(census_variables.keys()), # specify the variables we want, # specify the variables we want
    {'for': 'tract:*', # specify the geometry resolution we want
     'in': 'state:36'}, # specify the geometry bounds. In this case, the state of IL, Cook County
    year=2020, # specify the year
)
chi_tracts = pd.DataFrame(chi_tracts).rename(columns=census_variables)
chi_tracts['GEOID'] = chi_tracts['state'] + chi_tracts['county'] + chi_tracts['tract']
chi_tracts.head()

,Female,White,Black,Hispanic,Asian,MedIncome,TotalPop,Poverty,Bach,GradPro,...,Renter,Vacant,TotalUnits,Singleh,TotalHouse,MedHvalue,state,county,tract,GEOID
0,1715.0,3208.0,127.0,56.0,8.0,39659.0,3400.0,0.0,514.0,349.0,...,104.0,124.0,1381.0,1239.0,1381.0,186500.0,36,011,040600,36011040600
1,1883.0,3545.0,0.0,29.0,0.0,37904.0,3633.0,0.0,511.0,506.0,...,96.0,285.0,1768.0,1719.0,1768.0,174900.0,36,011,040700,36011040700
2,1956.0,3871.0,518.0,120.0,36.0,31871.0,4668.0,8.0,396.0,182.0,...,263.0,713.0,2241.0,1738.0,2241.0,147100.0,36,011,040800,36011040800
3,1753.0,3548.0,1.0,96.0,16.0,32996.0,3683.0,6.0,332.0,174.0,...,186.0,311.0,1767.0,1272.0,1767.0,142700.0,36,011,040900,36011040900
4,1409.0,2807.0,0.0,153.0,21.0,32727.0,3088.0,44.0,362.0,271.0,...,196.0,389.0,1581.0,1151.0,1581.0,147600.0,36,011,041001,36011041001


In [5]:
df.iloc[0]

Project ID                                          75039
Project Name                              ARCHER TOWERS 2
Project Start Date                             12/30/2025
Project Completion Date                               NaN
Building ID                                     1018491.0
Number                                             163-25
Street                                      ARCHER AVENUE
Borough                                            Queens
Postcode                                          11433.0
BBL                                          4101510065.0
BIN                                             4000000.0
Community Board                                     QN-12
Council District                                       27
Census Tract                                      44601.0
NTA - Neighborhood Tabulation Area                 QN1201
Latitude                                        40.703624
Longitude                                      -73.794864
Latitude (Inte

In [6]:
geometry =  gpd.points_from_xy(df['Longitude'], df['Latitude'])
crs = "EPSG:4326"
gdf = gpd.GeoDataFrame(df, 
                       geometry=geometry,
                       crs=crs)
gdf.head()

,Project ID,Project Name,Project Start Date,Project Completion Date,Building ID,Number,Street,Borough,Postcode,BBL,...,3-BR Units,4-BR Units,5-BR Units,6-BR+ Units,Unknown-BR Units,Counted Rental Units,Counted Homeownership Units,All Counted Units,Total Units,geometry
0,75039,ARCHER TOWERS 2,12/30/2025,NaN,1018491.0,163-25,ARCHER AVENUE,Queens,11433.0,4.101510e+09,...,0,0,0,0,0,400,0,400,400,POINT (-73.79486 40.70362)
1,77936,1060 PROSPECT AVENUE,12/29/2025,NaN,104255.0,1072,PROSPECT AVENUE,Bronx,10459.0,2.026910e+09,...,0,0,0,0,0,6,0,6,29,POINT (-73.89952 40.8247)
2,77935,121 MOUNT HOPE PLACE,12/25/2025,NaN,98083.0,121,MT HOPE PLACE,Bronx,10453.0,2.028050e+09,...,0,0,0,0,0,6,0,6,27,POINT (-73.90772 40.84874)
3,68309,SUTTER PLACE,12/24/2025,NaN,927579.0,2358,PITKIN AVENUE,Brooklyn,11207.0,3.040150e+09,...,0,0,0,0,0,26,0,26,26,POINT (-73.88538 40.67344)
4,68309,SUTTER PLACE,12/24/2025,NaN,927632.0,743,BLAKE AVENUE,Brooklyn,11207.0,3.037750e+09,...,8,0,0,0,0,8,0,8,8,POINT (-73.89123 40.66838)


In [7]:
# Let's add our young ages together
chi_tracts['Age2029'] = chi_tracts['Age2024'] + chi_tracts['Age2529']

# Let's get rid of values < 0, these are errors
chi_tracts[chi_tracts[census_variables.values()] < 0] = np.nan

# Finally, let's create a few percentage columns
def create_percent_col(col_name, df, totalpop='TotalPop'):
    """
    Add a column that divdes column name values by total population.
    Takes col_name and the dataframe as inputs. totalpop col name optional
    Outputs the dataframe with the new column.
    """
    df[f"percent_{col_name}"] = 100 * df[col_name] / df[totalpop]
    return df
    

columns = ['White', 'Black', 'Hispanic', 'Asian', 'Poverty', 'Bach', 'GradPro', \
           'Age2029','Age3034', 'DroveAlone', 'Transit', 'Commuters', 'Renter']

for col_name in columns:
    chi_tracts = create_percent_col(col_name, chi_tracts)

chi_tracts.head()

,Female,White,Black,Hispanic,Asian,MedIncome,TotalPop,Poverty,Bach,GradPro,...,percent_Asian,percent_Poverty,percent_Bach,percent_GradPro,percent_Age2029,percent_Age3034,percent_DroveAlone,percent_Transit,percent_Commuters,percent_Renter
0,1715.0,3208.0,127.0,56.0,8.0,39659.0,3400.0,0.0,514.0,349.0,...,0.235294,0.000000,15.117647,10.264706,10.676471,5.764706,37.823529,0.000000,44.000000,3.058824
1,1883.0,3545.0,0.0,29.0,0.0,37904.0,3633.0,0.0,511.0,506.0,...,0.000000,0.000000,14.065511,13.927883,13.239747,3.165428,44.976603,0.000000,49.628406,2.642444
2,1956.0,3871.0,518.0,120.0,36.0,31871.0,4668.0,8.0,396.0,182.0,...,0.771208,0.171380,8.483290,3.898886,10.732648,8.097686,29.991431,0.000000,35.818338,5.634105
3,1753.0,3548.0,1.0,96.0,16.0,32996.0,3683.0,6.0,332.0,174.0,...,0.434428,0.162911,9.014390,4.724409,10.371979,2.633723,39.478686,0.081455,48.384469,5.050231
4,1409.0,2807.0,0.0,153.0,21.0,32727.0,3088.0,44.0,362.0,271.0,...,0.680052,1.424870,11.722798,8.775907,10.589378,7.610104,42.940415,0.161917,54.371762,6.347150


In [8]:
# Lets add the geometry information
year=2020
state_fips='36'
url = f"https://www2.census.gov/geo/tiger/TIGER{year}/TRACT/tl_{year}_{state_fips}_tract.zip"
tracts = gpd.read_file(url)
tracts = tracts.to_crs(epsg=4326)
# And merge them on the GEOID 
chi_tracts = pd.merge(left=chi_tracts, right=tracts, on='GEOID', how='left')
chi_tracts = gpd.GeoDataFrame(chi_tracts)
chi_tracts.crs = "EPSG:4326"
chi_tracts.head()

,Female,White,Black,Hispanic,Asian,MedIncome,TotalPop,Poverty,Bach,GradPro,...,TRACTCE,NAME,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry
0,1715.0,3208.0,127.0,56.0,8.0,39659.0,3400.0,0.0,514.0,349.0,...,040600,406,Census Tract 406,G5020,S,74642568,61548,+42.9775784,-076.5183651,"POLYGON ((-76.57121 42.99702, -76.56942 42.997..."
1,1883.0,3545.0,0.0,29.0,0.0,37904.0,3633.0,0.0,511.0,506.0,...,040700,407,Census Tract 407,G5020,S,54082299,6719485,+42.8937652,-076.4932834,"POLYGON ((-76.55245 42.9264, -76.55028 42.9264..."
2,1956.0,3871.0,518.0,120.0,36.0,31871.0,4668.0,8.0,396.0,182.0,...,040800,408,Census Tract 408,G5020,S,175752178,13284270,+42.7812489,-076.4204198,"POLYGON ((-76.51535 42.84096, -76.5141 42.8409..."
3,1753.0,3548.0,1.0,96.0,16.0,32996.0,3683.0,6.0,332.0,174.0,...,040900,409,Census Tract 409,G5020,S,205918404,1572804,+42.6891179,-076.3568342,"POLYGON ((-76.4626 42.69178, -76.45628 42.6920..."
4,1409.0,2807.0,0.0,153.0,21.0,32727.0,3088.0,44.0,362.0,271.0,...,041001,410.01,Census Tract 410.01,G5020,S,208974937,9752821,+42.6849182,-076.5549870,"POLYGON ((-76.68983 42.67732, -76.68666 42.677..."


In [9]:
df_with_crs = gpd.sjoin(gdf, chi_tracts ,how='left')

In [10]:
df_with_crs

,Project ID,Project Name,Project Start Date,Project Completion Date,Building ID,Number,Street,Borough,Postcode,BBL,...,COUNTYFP,TRACTCE,NAME,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON
0,75039,ARCHER TOWERS 2,12/30/2025,NaN,1018491.0,163-25,ARCHER AVENUE,Queens,11433.0,4.101510e+09,...,081,044601,446.01,Census Tract 446.01,G5020,S,224776.0,0.0,+40.7049531,-073.7977905
1,77936,1060 PROSPECT AVENUE,12/29/2025,NaN,104255.0,1072,PROSPECT AVENUE,Bronx,10459.0,2.026910e+09,...,005,013100,131,Census Tract 131,G5020,S,230967.0,0.0,+40.8259541,-073.8978983
2,77935,121 MOUNT HOPE PLACE,12/25/2025,NaN,98083.0,121,MT HOPE PLACE,Bronx,10453.0,2.028050e+09,...,005,023301,233.01,Census Tract 233.01,G5020,S,119112.0,0.0,+40.8495612,-073.9087543
3,68309,SUTTER PLACE,12/24/2025,NaN,927579.0,2358,PITKIN AVENUE,Brooklyn,11207.0,3.040150e+09,...,047,115000,1150,Census Tract 1150,G5020,S,158250.0,0.0,+40.6739223,-073.8865224
4,68309,SUTTER PLACE,12/24/2025,NaN,927632.0,743,BLAKE AVENUE,Brooklyn,11207.0,3.037750e+09,...,047,115800,1158,Census Tract 1158,G5020,S,154981.0,0.0,+40.6688809,-073.8921624
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8978,55697,CONFIDENTIAL,01/14/2014,01/14/2014,NaN,'----,'----,Brooklyn,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8979,55773,CONFIDENTIAL,01/10/2014,01/10/2014,NaN,'----,'----,Staten Island,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8980,57341,CONFIDENTIAL,01/10/2014,01/10/2014,NaN,'----,'----,Staten Island,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8981,55647,CONFIDENTIAL,01/07/2014,01/07/2014,NaN,'----,'----,Brooklyn,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
df_with_crs.columns

Index(['Project ID', 'Project Name', 'Project Start Date',
       'Project Completion Date', 'Building ID', 'Number', 'Street', 'Borough',
       'Postcode', 'BBL', 'BIN', 'Community Board', 'Council District',
       'Census Tract', 'NTA - Neighborhood Tabulation Area', 'Latitude',
       'Longitude', 'Latitude (Internal)', 'Longitude (Internal)',
       'Building Completion Date', 'Reporting Construction Type',
       'Extended Affordability Only', 'Prevailing Wage Status',
       'Extremely Low Income Units', 'Very Low Income Units',
       'Low Income Units', 'Moderate Income Units', 'Middle Income Units',
       'Other Income Units', 'Studio Units', '1-BR Units', '2-BR Units',
       '3-BR Units', '4-BR Units', '5-BR Units', '6-BR+ Units',
       'Unknown-BR Units', 'Counted Rental Units',
       'Counted Homeownership Units', 'All Counted Units', 'Total Units',
       'geometry', 'index_right', 'Female', 'White', 'Black', 'Hispanic',
       'Asian', 'MedIncome', 'TotalPop', 'Pove

In [18]:
keep_cols = ['Project ID', 'Project Name', 'Project Start Date',
       'Project Completion Date', 'Building ID', 'Number', 'Street', 'Borough',
       'Postcode', 'BBL', 'BIN', 'Community Board', 'Council District',
       'Census Tract', 'NTA - Neighborhood Tabulation Area', 'Latitude',
       'Longitude', 'Latitude (Internal)', 'Longitude (Internal)',
       'Building Completion Date', 'Reporting Construction Type',
       'Extended Affordability Only', 'Prevailing Wage Status',
       'Extremely Low Income Units', 'Very Low Income Units',
       'Low Income Units', 'Moderate Income Units', 'Middle Income Units',
       'Other Income Units', 'Studio Units', '1-BR Units', '2-BR Units',
       '3-BR Units', '4-BR Units', '5-BR Units', '6-BR+ Units',
       'Unknown-BR Units', 'Counted Rental Units',
       'Counted Homeownership Units', 'All Counted Units', 'Total Units',
        'index_right', 'MedIncome', 'TotalPop', 'GEOID', 'percent_White',
       'percent_Black', 'percent_Hispanic', 'percent_Asian', 'percent_Poverty',
       'percent_Bach', 'percent_GradPro', 'percent_Age2029', 'percent_Age3034',
       'percent_Renter', 'ALAND', 'AWATER', 'geometry']

df_tokeep = df_with_crs[keep_cols]

In [25]:
df_tokeep.dropna().to_file('/Users/madilore/Documents/Cornell/Course Materials/QuantMethods/github_qm/crp_2100private/Week7/affordable_housing.shp')

/var/folders/p3/nl4g9f296vj7y7kcrk8f76380000gn/T/ipykernel_94557/2902075342.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  df_tokeep.dropna().to_file('/Users/madilore/Documents/Cornell/Course Materials/QuantMethods/github_qm/crp_2100private/Week7/affordable_housing.shp')
/Users/madilore/opt/miniconda3/envs/ius_2026/lib/python3.14/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'Project Name' to 'Project Na'
  ogr_write(
/Users/madilore/opt/miniconda3/envs/ius_2026/lib/python3.14/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'Project Start Date' to 'Project St'
  ogr_write(
/Users/madilore/opt/miniconda3/envs/ius_2026/lib/python3.14/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'Project Completion Date' to 'Project Co'
  ogr_write(
/Users/madilore/opt/miniconda3/envs/ius_2026/lib/python3.14/site-packages/pyogrio/raw.py:

In [20]:
df_tokeep.to_csv('/Users/madilore/Documents/Cornell/Course Materials/QuantMethods/2026/Week07/affordable_housing.csv', index=False)

In [21]:
df_tokeep.iloc[0]

Project ID                                                   75039
Project Name                                       ARCHER TOWERS 2
Project Start Date                                      12/30/2025
Project Completion Date                                        NaN
Building ID                                              1018491.0
Number                                                      163-25
Street                                               ARCHER AVENUE
Borough                                                     Queens
Postcode                                                   11433.0
BBL                                                   4101510065.0
BIN                                                      4000000.0
Community Board                                              QN-12
Council District                                                27
Census Tract                                               44601.0
NTA - Neighborhood Tabulation Area                          QN

In [22]:
df_tokeep.dropna()

,Project ID,Project Name,Project Start Date,Project Completion Date,Building ID,Number,Street,Borough,Postcode,BBL,...,percent_Asian,percent_Poverty,percent_Bach,percent_GradPro,percent_Age2029,percent_Age3034,percent_Renter,ALAND,AWATER,geometry
78,76012,479 CLINTON AVENUE HDFC.HPO.FY26,12/19/2025,12/19/2025,222886.0,477,CLINTON AVENUE,Brooklyn,11238.0,3.019770e+09,...,2.440966,0.000000,36.083842,26.532237,17.803131,22.393208,36.110374,188914.0,0.0,POINT (-73.96698 40.68402)
81,76928,MALCOLM X II PHASE B.HUDMF.FY26,12/19/2025,12/19/2025,41775.0,112,WEST 144 STREET,Manhattan,10030.0,1.020120e+09,...,1.586468,0.724942,9.970582,5.778525,18.092036,4.801429,33.525951,180488.0,0.0,POINT (-73.93741 40.82005)
82,77669,CAPITOL HALL.YR15.FY26,12/19/2025,12/19/2025,36448.0,166,WEST 87 STREET,Manhattan,10024.0,1.012170e+09,...,13.713777,0.000000,31.659888,34.354347,8.146924,12.747839,42.844433,178602.0,0.0,POINT (-73.97345 40.7879)
102,77041,HUNTERSMOON HALL.YR15.FY26,12/18/2025,12/18/2025,7858.0,2612,BROADWAY,Manhattan,10025.0,1.018700e+09,...,10.174297,0.000000,21.524570,33.819824,13.198236,4.462411,25.409492,218111.0,185382.0,POINT (-73.97054 40.79635)
115,77330,EXTRA PLACE.YR15.FY26,12/15/2025,12/15/2025,877389.0,4,EAST 1 STREET,Manhattan,10003.0,1.004578e+09,...,13.242424,0.000000,32.878788,15.030303,25.636364,18.393939,34.393939,89011.0,0.0,POINT (-73.99206 40.7248)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8967,50682,74 W 105th Street,01/21/2014,01/21/2014,37582.0,74,WEST 105 STREET,Manhattan,10025.0,1.018400e+09,...,11.267478,0.000000,12.338841,12.184492,15.807155,6.255675,36.762303,265404.0,0.0,POINT (-73.96268 40.79827)
8972,49722,Livonia Commons,01/15/2014,04/25/2016,948722.0,491,SHEFFIELD AVENUE,Brooklyn,11207.0,3.038058e+09,...,4.006918,0.000000,7.264341,4.093399,11.328913,7.783223,37.186509,168500.0,0.0,POINT (-73.8951 40.66499)
8973,49722,Livonia Commons,01/15/2014,04/25/2016,948723.0,494,SHEFFIELD AVE,Brooklyn,11207.0,3.038048e+09,...,4.006918,0.000000,7.264341,4.093399,11.328913,7.783223,37.186509,168500.0,0.0,POINT (-73.89511 40.66494)
8974,49722,Livonia Commons,01/15/2014,04/25/2016,948724.0,494,GEORGIA AVENUE,Brooklyn,11207.0,3.038208e+09,...,0.000000,0.341566,11.508145,4.466632,13.084603,9.301104,26.169207,193832.0,0.0,POINT (-73.89586 40.66417)
